In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import load_breast_cancer

In [ ]:
cancer=load_breast_cancer()

In [ ]:
x = cancer.data
y = cancer.target

In [ ]:
cancer.target_names

In [ ]:
df=pd.DataFrame(data=cancer.data,columns=cancer.feature_names)
df["target"]=cancer.target

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
sns.countplot(data=df, x='target' )

In [ ]:
sns.pairplot(df, hue='target')

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(20,20))  
sns.heatmap(df.corr(), cmap="coolwarm")

plt.xticks(rotation=90, fontsize=10)  
plt.yticks(rotation=0, fontsize=10)   
plt.show()

In [ ]:
df

In [ ]:
df.duplicated().sum()

In [ ]:
sns.scatterplot(data=df, x='mean radius', y='mean concavity',hue='target')

In [ ]:
x= df.drop('target', axis=1)
y= df['target']

In [ ]:
numeric_cols = x.columns


n_cols = 5
n_rows = 6

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(data=df, x=col, kde=True, bins=25, hue='target', palette='Set2', ax=axes[i])
    axes[i].set_title(col, fontsize=10)
    axes[i].grid(True)

plt.tight_layout()

In [ ]:
df = pd.concat([x, y], axis=1)

In [ ]:
cols_have_outliers = ['mean area', 'mean concavity', 'radius error', 'perimeter error', 'area error','compactness error', 'concavity error', 'worst area']

In [ ]:
Q1 = df[cols_have_outliers].quantile(0.25)
Q3 = df[cols_have_outliers].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


df_no_outliers = df[~((df[cols_have_outliers] < lower_bound) | (df[cols_have_outliers] > upper_bound)).any(axis=1)]


In [ ]:
df_no_outliers

In [ ]:
x= df.drop('target', axis=1)
y= df['target']

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train , y_test= train_test_split(x,y, test_size=0.2, random_state=42)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train= scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(x_train, y_train)
y_pred = lr.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix

lr_acc=accuracy_score(y_test, y_pred)

print(f'Accuracy = {lr_acc}')
print(sns.heatmap(confusion_matrix(y_test,y_pred), annot=True))
print(classification_report(y_test,y_pred))

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

model = LogisticRegression(max_iter=3000)
scores = cross_val_score(model, x, y, cv=5)  
print("Scores for each fold:", scores)
print("Mean Accuracy:", np.mean(scores))

# SVM

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV


svc= SVC()
parameters = {
    'kernel':['linear','rbf'],
    'gamma':[0.001, 0.01, 0.1],
    'C':[0.1, 1, 10]
}
grid_search = GridSearchCV(svc, parameters,refit=True, verbose=2)

In [ ]:
df= pd.concat([x,y], axis=1)

sample = df.sample(150)
features = sample.drop('target', axis=1)
label = sample['target']

grid_search.fit(features,label)

In [ ]:
grid_search.best_estimator_

In [ ]:
svc = SVC(kernel='linear',C=0.1, gamma=0.001)
svc.fit(x_train, y_train)
y_pred = svc.predict(x_test)

In [ ]:
svc_acc=accuracy_score(y_test, y_pred)
print(f'Accuracy = {svc_acc}')
print(sns.heatmap(confusion_matrix(y_test,y_pred), annot=True))
print(classification_report(y_test,y_pred))

In [ ]:
scores = cross_val_score(svc, x, y, cv=5)  
print("Scores for each fold:", scores)
print("Mean Accuracy:", np.mean(scores))

# KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score

k_values = list(range(1, 21))
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, x_train, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())



best_k = k_values[np.argmax(cv_scores)]
print(f"Best K value (via CV): {best_k} with average CV accuracy: {max(cv_scores):.2f}")

#Train final model with best_k and evaluate on test set
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(x_train, y_train)

# Predict and get accuracy on the test set
y_pred = final_knn.predict(x_test)
knn_acc = accuracy_score(y_test, y_pred)

print(f"Final Test Accuracy with K={best_k}: {knn_acc:.2f}")
print(classification_report(y_test,y_pred))

# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
dt.fit(x_train, y_train)
y_pred = dt.predict(x_test)

In [ ]:
dt_acc=accuracy_score(y_test, y_pred)
print(f'Accuracy = {dt_acc}')
print(sns.heatmap(confusion_matrix(y_test,y_pred), annot=True))
print(classification_report(y_test,y_pred))

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=20,random_state=0)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

In [ ]:
rf_acc=accuracy_score(y_test, y_pred)
print(f'Accuracy = {accuracy_score(y_test, y_pred)}')
print(sns.heatmap(confusion_matrix(y_test,y_pred), annot=True))
print(classification_report(y_test,y_pred))

# XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(random_state=42)

xgb.fit(x_train, y_train)
y_pred = xgb.predict(x_test)

In [ ]:
xgb_acc=accuracy_score(y_test, y_pred)
print(f'Accuracy = {accuracy_score(y_test, y_pred)}')
print(sns.heatmap(confusion_matrix(y_test,y_pred), annot=True))
print(classification_report(y_test,y_pred))

In [ ]:
models = pd.DataFrame({
    'Model':['Logistic Regression','SVC', 'KNN', 'Decision Tree Classifier', 'Random Forest Classifier','XGBoost'],
    'Score':[lr_acc*100,svc_acc*100,knn_acc*100,dt_acc*100,rf_acc*100,xgb_acc*100]
})
models.sort_values(by='Score', ascending=False)

In [ ]:
import pickle


model = knn


pickle.dump(model, open("breastC.pkl", "wb"))
